# LOCAL-26 验证记录（WIP）

这份 notebook 是对 LOCAL-26 验证日志的只读汇总。它只读取 `docs/validation/local26-20261012-*` 文件，不启动容器、不申请 GPU，也不重跑任何命令。GPU 和真实模型命令已在 sjq3 的 `dongyang` tmux 独立窗口执行，结果以原始日志为准。

当前记录覆盖依赖 gate、CPU 单测、GPU smoke、真实模型、启动器、单测、静态检查和 adapter 产物核验。

## 运行记录

| 项目 | 运行位置 | 归档命令 | 原始日志 | 通过条件 |
|---|---|---|---|---|
| 依赖 gate | sjq3 `dongyang` tmux，CPU 容器 | `commands/local26-deps-gate.sh` | `local26-20261012-dependency-gate.txt` | 真实镜像安装后无新增冲突；两个负例被拒绝；退出码 0 |
| CPU runner | sjq3 `dongyang` tmux | `commands/local26-cpu.sh` | `local26-20261012-cpu.log` | torchada 版本和 runner 测试通过 |
| GPU smoke | sjq3 `dongyang` tmux，S5000 / `musa:0` | `commands/local26-gpu.sh` | `local26-20261012-gpu-smoke.log` | float32、bf16 均完成前向、反向、生成、LoRA、保存重载 |
| 真实模型 | sjq3 `dongyang` tmux，S5000 / `musa:0` | `commands/local26-model.sh` | `local26-20261012-model.log` | Qwen3/Qwen2 各完成 infer 和 LoRA 2 steps |
| 启动器 | sjq3 `dongyang` tmux，显式 `GPU_ALLOW_UNLEASED=1` | `commands/local26-launcher-check.sh` | `local26-20261012-launcher.log` | infer 成功且 `LAUNCHER_CHECK_EXIT=0` |
| 启动器/runner 单测 | macOS `dongyang` tmux | 仓库测试命令 | `local26-20261012-tests.log` | 11 项通过，退出码 0 |
| 静态检查 | macOS `dongyang` tmux | `commands/local26-static.py` | `local26-20261012-static.json` | compileall、bash -n、diff --check 均为 0 |
| adapter 产物 | sjq3 `dongyang` tmux，CPU 加载权重核验 | `commands/local26-artifact.sh` | `local26-20261012-artifact.log` | 两个 adapter 的 safetensors 和非零 LoRA B 均通过 |

### 实际命令

- GPU smoke：`python scripts/musa/run.py scripts/musa/smoke.py --dtype float32` 和 `--dtype bf16`。
- 真实模型：`qwen.py infer` 与 `qwen.py lora --max-steps 2`，模型为 `Qwen3-0.6B`、`Qwen2-0.5B-Instruct`，dtype 为 `bf16`。
- 启动器：`GPU_ALLOW_UNLEASED=1 GPU_INDICES=0 MUSA_MODELS_ROOT=/data/models bash scripts/musa/container.sh ...`。该变量只用于本次启动器路径验证，日志已明确记录。

In [7]:
# 展示当时在 tmux 执行的脚本；这里只读文件，不调用 subprocess。
from pathlib import Path
for name in ("local26-cpu.sh", "local26-deps-gate.sh", "local26-gpu.sh", "local26-model.sh", "local26-launcher-check.sh", "local26-artifact.sh", "local26-verify-artifacts.py", "local26-static.py"):
    path = Path("docs/validation/commands") / name
    print("\n=== " + name + " ===")
    print(path.read_text())
print("macOS tmux tests: python3 -m unittest discover -s scripts/musa/tests -v")



=== local26-cpu.sh ===
#!/usr/bin/env bash
set -o pipefail
python3 - <<'PY'
from pathlib import Path
for p in Path('/home/mccxadmin/self-llm-local26/scripts/musa').rglob('._*'):
    if p.is_file():
        p.unlink()
PY
docker run --rm --runtime=runc -e MTHREADS_VISIBLE_DEVICES=none \
  --label serverlist.owner=dongyang --label serverlist.ticket=LOCAL-26 \
  --mount type=bind,src=/home/mccxadmin/self-llm-local26,dst=/workspace/self-llm \
  --workdir /workspace/self-llm \
  registry.mthreads.com/mcctest/training-suite:v2.1.7 bash -lc '
    python -m pip install --no-deps torchada &&
    python -c "import importlib.metadata as m; print(\"TORCHADA_VERSION=\" + m.version(\"torchada\"))" &&
    python -m unittest discover -s scripts/musa/tests -v &&
    python -m compileall -q scripts/musa &&
    bash -n scripts/musa/install.sh scripts/musa/container.sh
  ' 2>&1 | tee /tmp/local26-cpu-check.log
printf 'CPU_CHECK_EXIT=%s\n' "${PIPESTATUS[0]}"


=== local26-deps-gate.sh ===
#!/usr/bin/env ba

In [1]:
from pathlib import Path
import json
import re
from datetime import datetime

ROOT = Path.cwd()
if ROOT.name != "self-llm":
    for candidate in (ROOT, *ROOT.parents):
        if (candidate / "docs" / "validation").is_dir():
            ROOT = candidate
            break
LOG_DIR = ROOT / "docs" / "validation"
print("ROOT=", ROOT)
print("LOG_DIR=", LOG_DIR)
print("READ_AT=", datetime.now().isoformat(timespec="seconds"))


ROOT= /Users/dongyang/Documents/ChatGPT/self-llm
LOG_DIR= /Users/dongyang/Documents/ChatGPT/self-llm/docs/validation
READ_AT= 2026-10-12T23:20:01


In [2]:
EXPECTED_LOGS = {
    "dependency_gate": LOG_DIR / "local26-20261012-dependency-gate.txt",
    "cpu": LOG_DIR / "local26-20261012-cpu.log",
    "gpu_smoke": LOG_DIR / "local26-20261012-gpu-smoke.log",
    "model": LOG_DIR / "local26-20261012-model.log",
    "launcher": LOG_DIR / "local26-20261012-launcher.log",
    "tests": LOG_DIR / "local26-20261012-tests.log",
    "static": LOG_DIR / "local26-20261012-static.json",
    "artifact": LOG_DIR / "local26-20261012-artifact.log",
}

def read_log(path):
    return path.read_text(encoding="utf-8", errors="replace") if path.exists() else None

inventory = {
    name: {"path": str(path), "exists": path.exists(), "bytes": path.stat().st_size if path.exists() else None}
    for name, path in EXPECTED_LOGS.items()
}
print(json.dumps(inventory, ensure_ascii=False, indent=2))


{
  "dependency_gate": {
    "path": "/Users/dongyang/Documents/ChatGPT/self-llm/docs/validation/local26-20261012-dependency-gate.txt",
    "exists": true,
    "bytes": 2196
  },
  "cpu": {
    "path": "/Users/dongyang/Documents/ChatGPT/self-llm/docs/validation/local26-20261012-cpu.log",
    "exists": true,
    "bytes": 1006
  },
  "gpu_smoke": {
    "path": "/Users/dongyang/Documents/ChatGPT/self-llm/docs/validation/local26-20261012-gpu-smoke.log",
    "exists": true,
    "bytes": 6265
  },
  "model": {
    "path": "/Users/dongyang/Documents/ChatGPT/self-llm/docs/validation/local26-20261012-model.log",
    "exists": true,
    "bytes": 9052
  },
  "launcher": {
    "path": "/Users/dongyang/Documents/ChatGPT/self-llm/docs/validation/local26-20261012-launcher.log",
    "exists": true,
    "bytes": 3817
  },
  "tests": {
    "path": "/Users/dongyang/Documents/ChatGPT/self-llm/docs/validation/local26-20261012-tests.log",
    "exists": true,
    "bytes": 1048
  },
  "static": {
    "path": 

In [3]:
# 完整打印原始日志；不裁剪、不重跑。
for name, path in EXPECTED_LOGS.items():
    text = read_log(path)
    print(f"\n===== {name}: {path.name} =====")
    print(text if text is not None else "PENDING: 原始日志不存在")



===== dependency_gate: local26-20261012-dependency-gate.txt =====
Image dependency conflicts already present before installation:
ml-dtypes 0.6.0 has requirement numpy>=2.0.0, but you have numpy 1.26.4.
vllm 0.10.1.2.dev0+g1da94e673.d20260902.empty has requirement numpy==1.26.1, but you have numpy 1.26.4.
vllm 0.10.1.2.dev0+g1da94e673.d20260902.empty has requirement outlines_core==0.2.10, but you have outlines-core 0.1.26.
vllm 0.10.1.2.dev0+g1da94e673.d20260902.empty has requirement xgrammar==0.1.21; platform_machine == "x86_64" or platform_machine == "aarch64" or platform_machine == "arm64", but you have xgrammar 0.1.27.
vllm-musa 0.1.dev402+g3bad2b9.d20260902 has requirement pyarrow==20.0.0, but you have pyarrow 22.0.0.
{"baseline_conflicts": 5}
Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
{"baseline_conflicts": 5, "current_conflicts": 5, "new_conflicts": []}
REAL_IMAGE_GATE_EXIT=0
{"baseline_conflicts": 5, "current_conflicts": 6, "new_conflicts": ["local26-conflict

## 退出码和结构化结果

下面的汇总只使用日志中已经出现的显式标记。`dependency_gate` 保留镜像原有的 5 项冲突；这表示“无新增冲突”，不是依赖环境完全干净。两个 fixture 负例分别验证了新增冲突和 malformed metadata 会被拒绝。

In [4]:
EXIT_RE = re.compile(r"(?P<key>[A-Z][A-Z0-9_]*EXIT)\s*=\s*(?P<value>-?\d+)")
STATUS_RE = re.compile(r"(?i)\b(PASS(?:ED)?|FAIL(?:ED)?|BLOCKED|PENDING)\b")

def evidence(name, path):
    text = read_log(path)
    if text is None:
        return {"name": name, "file": path.name, "status": "PENDING", "exit_markers": [], "status_words": []}
    exits = [{"key": m.group("key"), "value": int(m.group("value"))} for m in EXIT_RE.finditer(text)]
    words = sorted({m.group(1).upper() for m in STATUS_RE.finditer(text)})
    return {"name": name, "file": path.name, "status": "AVAILABLE", "exit_markers": exits, "status_words": words}

evidence_rows = [evidence(name, path) for name, path in EXPECTED_LOGS.items()]
print(json.dumps(evidence_rows, ensure_ascii=False, indent=2))


[
  {
    "name": "dependency_gate",
    "file": "local26-20261012-dependency-gate.txt",
    "status": "AVAILABLE",
    "exit_markers": [
      {
        "key": "REAL_IMAGE_GATE_EXIT",
        "value": 0
      },
      {
        "key": "DEPENDENCY_GATE_EXIT",
        "value": 0
      }
    ],
    "status_words": [
      "FAILED"
    ]
  },
  {
    "name": "cpu",
    "file": "local26-20261012-cpu.log",
    "status": "AVAILABLE",
    "exit_markers": [],
    "status_words": []
  },
  {
    "name": "gpu_smoke",
    "file": "local26-20261012-gpu-smoke.log",
    "status": "AVAILABLE",
    "exit_markers": [
      {
        "key": "GPU_CHECK_EXIT",
        "value": 0
      }
    ],
    "status_words": [
      "PASS"
    ]
  },
  {
    "name": "model",
    "file": "local26-20261012-model.log",
    "status": "AVAILABLE",
    "exit_markers": [
      {
        "key": "MODEL_CHECK_EXIT",
        "value": 0
      }
    ],
    "status_words": [
      "PASS"
    ]
  },
  {
    "name": "launcher",
    

In [5]:
# 提取日志中的 JSON 结果行，便于 review 时核对模型、设备和 adapter 数量。
def json_lines(text):
    rows = []
    if not text:
        return rows
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("{") and line.endswith("}"):
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError:
                pass
    return rows

gpu_rows = json_lines(read_log(EXPECTED_LOGS["gpu_smoke"]))
model_rows = json_lines(read_log(EXPECTED_LOGS["model"]))
artifact_rows = json_lines(read_log(EXPECTED_LOGS["artifact"]))
print("GPU JSON results:", json.dumps(gpu_rows, ensure_ascii=False, indent=2))
print("MODEL JSON results:", json.dumps(model_rows, ensure_ascii=False, indent=2))
print("ARTIFACT JSON results:", json.dumps(artifact_rows, ensure_ascii=False, indent=2))


GPU JSON results: [
  {
    "baseline_conflicts": 5
  },
  {
    "baseline_conflicts": 5,
    "current_conflicts": 5,
    "new_conflicts": []
  },
  {
    "test": "torchada_musa_qwen2_peft_smoke",
    "status": "PASS",
    "versions": {
      "python": "3.10.12",
      "torchada": "0.1.96",
      "torch_musa": "2.7.1.post1+b6d53a2",
      "torch": "2.7.1.post1",
      "transformers": "4.57.0",
      "peft": "0.20.0"
    },
    "device": {
      "requested": "cuda:0",
      "name": "MTT S5000",
      "physical_type": "musa",
      "physical_index": 0
    },
    "dtype": "float32",
    "base_loss": 4.834263324737549,
    "base_gradient_tensors": 27,
    "generated_token_ids": [
      121,
      51,
      25,
      33,
      43,
      113,
      15,
      33,
      114,
      33,
      114,
      33
    ],
    "lora_loss": 4.506847381591797,
    "lora_trainable_parameters": 1792,
    "adapter_directory": "temporary (removed after successful reload)",
    "checks": {
      "physical_musa":

## Review 结论

- **依赖 gate：PASS（有边界）**。真实镜像安装 `torchada 0.1.96` 后仍为基线 5 项冲突，`new_conflicts=[]`；新增冲突和 malformed metadata 两个负例均被拒绝，`DEPENDENCY_GATE_EXIT=0`。
- **GPU smoke：PASS**。S5000 的 `musa:0` 上 float32、bf16 均通过，日志中的 `GPU_CHECK_EXIT=0`；两种 dtype 都覆盖 CUDA API、基础训练、greedy generation、LoRA 训练和 adapter 保存重载。
- **真实模型：PASS**。Qwen3-0.6B 和 Qwen2-0.5B-Instruct 均在 `musa:0` 完成 infer 和 LoRA 2 steps，分别产出 `/results/qwen3-lora`、`/results/qwen2-lora`，`MODEL_CHECK_EXIT=0`。
- **启动器和单测：PASS**。启动器 infer 通过且 `LAUNCHER_CHECK_EXIT=0`；容器/runner 共 11 项测试通过且 `TEST_CHECK_EXIT=0`。
- **静态检查：PASS**。compileall、bash -n、git diff --check 的记录均为退出码 0。
- **产物核验：PASS**。`qwen3-lora` 保存 112 个 tensor，其中 56 个非零 LoRA B；`qwen2-lora` 保存 96 个 tensor，其中 48 个非零 LoRA B；`ARTIFACT_CHECK_EXIT=0`。

剩余风险是镜像原有的 5 项依赖冲突（numpy、outlines-core、xgrammar、pyarrow 版本约束）以及日志中的弃用/缺少 config 警告。它们没有在本次安装后新增，但需要在后续镜像升级或依赖收敛时单独处理。notebook 本身没有再次执行 GPU。

In [6]:
summary = {
    row["name"]: {
        "log": row["file"],
        "status": ("PASS" if any(item["value"] == 0 for item in row["exit_markers"]) else row["status"]),
        "exit_markers": row["exit_markers"],
    }
    for row in evidence_rows
}
print(json.dumps(summary, ensure_ascii=False, indent=2))


{
  "dependency_gate": {
    "log": "local26-20261012-dependency-gate.txt",
    "status": "PASS",
    "exit_markers": [
      {
        "key": "REAL_IMAGE_GATE_EXIT",
        "value": 0
      },
      {
        "key": "DEPENDENCY_GATE_EXIT",
        "value": 0
      }
    ]
  },
  "cpu": {
    "log": "local26-20261012-cpu.log",
    "status": "AVAILABLE",
    "exit_markers": []
  },
  "gpu_smoke": {
    "log": "local26-20261012-gpu-smoke.log",
    "status": "PASS",
    "exit_markers": [
      {
        "key": "GPU_CHECK_EXIT",
        "value": 0
      }
    ]
  },
  "model": {
    "log": "local26-20261012-model.log",
    "status": "PASS",
    "exit_markers": [
      {
        "key": "MODEL_CHECK_EXIT",
        "value": 0
      }
    ]
  },
  "launcher": {
    "log": "local26-20261012-launcher.log",
    "status": "PASS",
    "exit_markers": [
      {
        "key": "LAUNCHER_CHECK_EXIT",
        "value": 0
      }
    ]
  },
  "tests": {
    "log": "local26-20261012-tests.log",
    "stat

## 2026-10-13：通用 Trainer 的 CPU 索引采样修复

旧修复只在 qwen.py 的专用 Trainer 中。本次将 CPU 采样的 Generator 和默认设备约束放到启动层，普通 Transformers Trainer 不需要改写采样器。Torchada 的全局默认设备 getter 返回 musa:0，默认 randperm 索引却在 CPU。

Qwen3-0.6B 两步 BF16 LoRA 通过，设备 musa:0，112 个 adapter 张量更新，loss=6.338409185409546。输入是人工 token 序列，只核对训练路径。

第一次检查误以为默认 RandomSampler 必然失败，提前断言退出，未进入训练；第二次实际复现 Accelerate 创建生成器后的 CPU/MUSA 错误。最终同时约束 CPU 采样的生成器和设备后通过。下面保留三次原始记录。


In [8]:
from pathlib import Path
import json
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate/"scripts/musa/run.py").exists():
        repository=candidate
        break
else:
    raise RuntimeError("在 self-llm 仓库内打开 Notebook")
root=repository/"docs/validation"
for name in ("commands/local26-stock-trainer.sh", "local26-20261013-stock-trainer-initial.log", "local26-20261013-stock-trainer-generator-failure.log", "local26-20261013-stock-trainer.log"):
    print("\n=== "+name+" ===")
    print((root/name).read_text())
print("\n=== macOS static checks ===")
print(json.dumps(json.loads((root/"local26-20261013-static.json").read_text()),ensure_ascii=False,indent=2))



=== commands/local26-stock-trainer.sh ===
#!/usr/bin/env bash
set -uo pipefail
cd /home/mccxadmin/self-llm-local26
GPU_ALLOW_UNLEASED=1 GPU_INDICES=0 MUSA_MODELS_ROOT=/data/models bash scripts/musa/container.sh bash -lc '
  set -euo pipefail
  bash scripts/musa/install.sh
  python scripts/musa/run.py scripts/musa/tests/stock_trainer_gpu.py
' 2>&1 | tee /tmp/local26-stock-trainer.log
code=${PIPESTATUS[0]}
echo "STOCK_TRAINER_CHECK_EXIT=$code" | tee -a /tmp/local26-stock-trainer.log
exit "$code"


=== local26-20261013-stock-trainer-initial.log ===
GPU lease checks bypassed by explicit GPU_ALLOW_UNLEASED=1
Image dependency conflicts already present before installation:
ml-dtypes 0.6.0 has requirement numpy>=2.0.0, but you have numpy 1.26.4.
vllm 0.10.1.2.dev0+g1da94e673.d20260902.empty has requirement numpy==1.26.1, but you have numpy 1.26.4.
vllm 0.10.1.2.dev0+g1da94e673.d20260902.empty has requirement outlines_core==0.2.10, but you have outlines-core 0.1.26.
vllm 0.10.1.2.dev0+g1da94e6